## CoreDNS revisit — customising cluster DNS

You met CoreDNS in notebook 04 — a Deployment in `kube-system`, fronted by a Service. The default config is fine for 95% of clusters; here are the cases that warrant a change.

### The Corefile

CoreDNS's config lives in a ConfigMap named `coredns` (`kubectl edit configmap coredns -n kube-system`). Each line is a **plugin**:

```
.:53 {
    kubernetes cluster.local ...   # source of all Service DNS (reads the API server)
    forward . /etc/resolv.conf     # non-cluster names → node's upstream resolver
    cache 30                        # cache positive answers 30s
    reload                          # re-read on ConfigMap change (no pod restart)
}
```

### Forward a custom domain

Pods needing an internal private hostname (`*.corp.internal`) — add a server block:

```
corp.internal:53 {
    forward . 10.0.0.10
    cache 30
}
```

Save the ConfigMap; the `reload` plugin picks it up.

### Per-pod DNS

One pod needing different DNS uses `dnsConfig`:

```yaml
spec:
  dnsPolicy: None
  dnsConfig:
    nameservers: [8.8.8.8]
    options: [{ name: ndots, value: "2" }]
```

`dnsPolicy`: `ClusterFirst` (default — CoreDNS), `Default` (node's resolv.conf), `None` (only `dnsConfig`), `ClusterFirstWithHostNet` (hostNetwork pods).

### The `ndots` trap

Default `ndots: 5` means any name with <5 dots tries every search domain first — `curl google.com` fires four failing cluster lookups before the real one. Perf-sensitive workloads set `ndots: 2`. This closes the networking module — model, CNI, paths, Ingress, policy, DNS. On our map it's the **CoreDNS** chip in the Networking box, the name layer we've now customised end to end.